In [2]:
import numpy as np
import torch
import torch.nn as nn

In [3]:
n_batch = 4
n_type = 2
MPC_T = 10
T = 6
n_state = 5
n_ctrl = 1
data_batch = []
for i in range(n_batch):
    data_type = []
    for j in range(n_type):
        data_mpc = []
        for k in range(MPC_T):
            if j == 0:
                data_mpc.append(np.random.rand(n_state, T))
            else:
                data_mpc.append(np.random.rand(n_ctrl, T))
        data_type.append(data_mpc)
    data_batch.append(data_type)

In [4]:
print(len(data_batch))
print(len(data_batch[0]))

4
2


In [9]:
print(len(data_batch[:]))

4


In [47]:
results_states = torch.tensor(np.array([result[0] for result in data_batch]), dtype=torch.float32) # (n_batch, MPC_T, n_state, T)
results_inputs = torch.tensor(np.array([result[1] for result in data_batch]), dtype=torch.float32) # (n_batch, MPC_T, n_ctrl, T)
print(results_states.shape)
print(results_inputs.shape)

torch.Size([4, 10, 5, 6])
torch.Size([4, 10, 1, 6])


In [48]:
x_list_ori = results_states[:, :, :, 0] # (n_batch, MPC_T, n_state)
u_list_ori = results_inputs[:, :, :, 0] # (n_batch, MPC_T, n_ctrl)
print(x_list_ori.shape)
print(u_list_ori.shape)

torch.Size([4, 10, 5])
torch.Size([4, 10, 1])


In [49]:
x_list = x_list_ori.permute(1, 0, 2)
u_list = u_list_ori.permute(1, 0, 2)
print(x_list.shape)
print(u_list.shape)

torch.Size([10, 4, 5])
torch.Size([10, 4, 1])


In [61]:
def cost_cartpole(x, u, Q, R, Qf, is_terminal):
    """
    x shape: (n_batch, n_state, 1)
    u shape: (n_batch, n_ctrl, 1)
    """
    if is_terminal:
        cost = torch.matmul(torch.matmul(x.transpose(1, 2), Qf), x)
    else:
        cost = torch.matmul(torch.matmul(x.transpose(1, 2), Q), x) + torch.matmul(torch.matmul(u.transpose(1, 2), R), u)
    return cost.squeeze(-1).squeeze(-1).squeeze(-1)

test_x = torch.rand(n_batch, n_state, 1)
test_u = torch.rand(n_batch, n_ctrl, 1)
Q = torch.eye(n_state)
R = torch.eye(n_ctrl)
Qf = torch.eye(n_state)
print(Q.dtype)
print(cost_cartpole(test_x, test_u, Q, R, Qf, False))

torch.float32
tensor([1.9630, 1.8009, 2.4898, 1.4916])


In [62]:
print(results_states[:, 0, :, 0].shape)
print(results_states[:, 0, :, 0].dtype)
cost_cartpole(results_states[:, 0, :, 0].unsqueeze(2), results_inputs[:, 0, :, 0].unsqueeze(2), Q, R, Qf, False)

torch.Size([4, 5])
torch.float32


tensor([1.4077, 1.7313, 1.5539, 1.8484])

In [63]:
def VN_cartpole_multi(results_states, results_inputs, Q, R, Qf):
    n_batch, MPC_T, n_state, T = results_states.shape

    VN_list = []
    for n in range(MPC_T):
        VN = 0
        for t in range(T):
            if t == T - 1:
                VN += cost_cartpole(results_states[:, n, :, t].unsqueeze(2), results_inputs[:, n, :, t].unsqueeze(2), Q, R, Qf, True)
            else:
                VN += cost_cartpole(results_states[:, n, :, t].unsqueeze(2), results_inputs[:, n, :, t].unsqueeze(2), Q, R, Qf, False)
        VN_list.append(VN)

    return VN_list

VN_list = VN_cartpole_multi(results_states, results_inputs, Q, R, Qf)
print(VN_list)

[tensor([12.3556, 11.8551,  9.0369, 12.0002]), tensor([10.3035, 10.5163, 16.6232, 11.3859]), tensor([12.1109, 11.1129, 10.2010, 12.2630]), tensor([12.2182, 12.3988,  8.9867, 12.2296]), tensor([ 9.8860, 11.8845, 12.5666, 13.6844]), tensor([ 9.2075, 11.2695, 15.0023,  8.7398]), tensor([11.2391, 11.3447,  8.8070, 13.1466]), tensor([10.5742, 11.2364, 10.0168, 13.3962]), tensor([11.0573, 11.0213, 10.1561, 11.9316]), tensor([11.9354, 12.3271, 12.4890, 10.5887])]


In [67]:
def RDP_criteria_cartpole(VN_list, x_list, u_list, alpha, Q, R, Qf, MPC_T, func, test=False, log_path=None):
    loss = 0
    for i in range(MPC_T - 1):
        RDP = (VN_list[i+1] + alpha * cost_cartpole(x_list[i].unsqueeze(2), u_list[i].unsqueeze(2), Q, R, Qf, False)) - VN_list[i] # Wish RDP <= 0

        if test:
            if log_path is not None:
                with open(log_path, 'a') as f:
                    f.write(f'RDP{i}: {RDP}\n')
        loss += func(RDP)
    return loss

Q = nn.Parameter(torch.eye(5))
R = torch.eye(1)
F = nn.Parameter(torch.eye(5))
print(RDP_criteria_cartpole(VN_list, x_list, u_list, 1, Q, R, F, MPC_T, lambda x: torch.relu(x), test=True, log_path=None))
print(RDP_criteria_cartpole(VN_list, x_list, u_list, 1, Q, R, F, MPC_T, lambda x: torch.relu(x), test=True, log_path=None).mean())

tensor([19.3437, 17.7847, 30.2115, 18.9876], grad_fn=<AddBackward0>)
tensor(21.5819, grad_fn=<MeanBackward0>)
